# Card Mask Builder

Build a reusable binary card mask from `L1000768_red_card_02.png` using red thresholding, min-area cleanup, and closing.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("ipywidgets is required. Install it with: pip install ipywidgets") from exc

BASE_DIR = Path.cwd()
CARD_PATH = BASE_DIR / "iapr-26-uno-vision-challenge" / "cards" / "L1000768_red_card_02.png"
RED_HSV_PATH = BASE_DIR / "red_hsv.txt"
OUTPUT_MASK_PATH = BASE_DIR / "general_card_mask.png"
MASK_WIDTH = 75
MASK_HEIGHT = 125
HSV_ARRAY_PATTERN = re.compile(r"np\.array\(\s*\[([^\]]+)\]\s*\)")


In [ ]:
def parse_hsv_file(path: Path) -> list[tuple[np.ndarray, np.ndarray]]:
    arrays = []
    for match in HSV_ARRAY_PATTERN.finditer(path.read_text(encoding="utf-8")):
        values = [int(value.strip()) for value in match.group(1).split(",")]
        arrays.append(np.array(values, dtype=np.uint8))
    if len(arrays) % 2 != 0 or not arrays:
        raise ValueError(f"Expected lower/upper HSV pairs in {path}")
    return list(zip(arrays[0::2], arrays[1::2]))


def threshold_hsv_image(image_bgr: np.ndarray, ranges: list[tuple[np.ndarray, np.ndarray]]) -> np.ndarray:
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lower, upper in ranges:
        mask = cv2.bitwise_or(mask, cv2.inRange(hsv, lower, upper))
    return mask


def filter_small_components(mask: np.ndarray, min_area: int) -> np.ndarray:
    if min_area <= 0:
        return mask.copy()
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    filtered = np.zeros_like(mask)
    for label in range(1, num_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            filtered[labels == label] = 255
    return filtered


def close_mask(mask: np.ndarray, close_size: int, close_iter: int) -> np.ndarray:
    if close_size <= 0 or close_iter <= 0:
        return mask.copy()
    if close_size % 2 == 0:
        close_size += 1
    kernel = np.ones((close_size, close_size), dtype=np.uint8)
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=close_iter)



def normalize_card_mask(mask: np.ndarray, width: int = MASK_WIDTH, height: int = MASK_HEIGHT) -> np.ndarray:
    points = cv2.findNonZero(mask)
    if points is None:
        return np.zeros((height, width), dtype=np.uint8)
    x, y, w, h = cv2.boundingRect(points)
    cropped = mask[y:y + h, x:x + w]
    normalized = cv2.resize(cropped, (width, height), interpolation=cv2.INTER_NEAREST)
    return cv2.threshold(normalized, 127, 255, cv2.THRESH_BINARY)[1]


def build_card_mask(image_bgr: np.ndarray, ranges: list[tuple[np.ndarray, np.ndarray]], min_area: int, close_size: int, close_iter: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    raw_mask = threshold_hsv_image(image_bgr, ranges)
    area_mask = filter_small_components(raw_mask, min_area)
    final_mask = close_mask(area_mask, close_size, close_iter)
    return raw_mask, area_mask, final_mask


In [ ]:
image_bgr = cv2.imread(str(CARD_PATH))
if image_bgr is None:
    raise FileNotFoundError(f"Could not load card image: {CARD_PATH}")
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
red_ranges = parse_hsv_file(RED_HSV_PATH)

print(f"Card image: {CARD_PATH}")
print(f"Red HSV settings: {RED_HSV_PATH}")
print(f"Output mask: {OUTPUT_MASK_PATH}")


In [ ]:
min_area_slider = widgets.IntSlider(value=0, min=0, max=50000, step=50, description="Min area")
close_size_slider = widgets.IntSlider(value=5, min=0, max=51, step=2, description="Close size")
close_iter_slider = widgets.IntSlider(value=1, min=0, max=50, step=1, description="Close iter")
save_button = widgets.Button(description="Save", button_style="success")
status = widgets.Output()
preview = widgets.Output()
latest_mask = None


def update_preview(*_):
    global latest_mask
    raw_mask, area_mask, final_mask = build_card_mask(
        image_bgr,
        red_ranges,
        min_area_slider.value,
        close_size_slider.value,
        close_iter_slider.value,
    )
    normalized_mask = normalize_card_mask(final_mask)
    latest_mask = normalized_mask
    masked_preview = cv2.bitwise_and(image_rgb, image_rgb, mask=final_mask)

    with preview:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 5, figsize=(20, 5))
        axes[0].imshow(image_rgb)
        axes[0].set_title("card")
        axes[1].imshow(raw_mask, cmap="gray")
        axes[1].set_title("red threshold")
        axes[2].imshow(final_mask, cmap="gray")
        axes[2].set_title("general card mask")
        axes[3].imshow(normalized_mask, cmap="gray")
        axes[3].set_title("saved 75x125 mask")
        axes[4].imshow(masked_preview)
        axes[4].set_title("masked card")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    with status:
        clear_output(wait=True)
        print(f"Min area: {min_area_slider.value}")
        print(f"Close size: {close_size_slider.value}")
        print(f"Close iter: {close_iter_slider.value}")
        print(f"Mask pixels: {int(np.count_nonzero(final_mask))}")
        print(f"Saved mask size: {latest_mask.shape[1]}x{latest_mask.shape[0]}")


def save_mask(_):
    if latest_mask is None:
        update_preview()
    if not cv2.imwrite(str(OUTPUT_MASK_PATH), latest_mask):
        raise ValueError(f"Could not write mask: {OUTPUT_MASK_PATH}")
    with status:
        print(f"Saved general card mask to {OUTPUT_MASK_PATH}")


for widget in (min_area_slider, close_size_slider, close_iter_slider):
    widget.observe(update_preview, names="value")

save_button.on_click(save_mask)

display(widgets.VBox([
    widgets.HBox([min_area_slider, close_size_slider, close_iter_slider]),
    save_button,
    status,
    preview,
]))

update_preview()
